# 01 — Exploração e Análise do Dataset

**Objetivo:** Entender a estrutura e distribuição do dataset de tomografias pulmonares antes de qualquer modelagem.

## Dataset: Chest CT-Scan Images

O dataset contém imagens de tomografia computadorizada do tórax organizadas em **4 classes**:

| Classe | Descrição |
|--------|----------|
| `adenocarcinoma` | Tipo mais comum de câncer de pulmão (origem glandular) |
| `large.cell.carcinoma` | Carcinoma de células grandes (crescimento rápido) |
| `squamous.cell.carcinoma` | Carcinoma epidermoide (origem nas células escamosas) |
| `normal` | Pulmão saudável, sem malignidade |

**Divisão:** `train` / `valid` / `test` — já pré-definida no dataset.

## 1. Configuração Inicial

In [ ]:
# Adiciona o diretório src ao path para importar os módulos do projeto
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

# Bibliotecas padrão
import os
import random
from collections import Counter

# Processamento e visualização
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image

# Configurações de estilo para os gráficos
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='Set2')

# Módulo de configuração do projeto
from config import (
    CAMINHO_TREINO, CAMINHO_TESTE, CAMINHO_VALIDACAO,
    NOMES_CLASSES, MAPA_CLASSES, SEMENTE_ALEATORIA
)

random.seed(SEMENTE_ALEATORIA)
np.random.seed(SEMENTE_ALEATORIA)

print('Configuração carregada com sucesso!')

## 2. Estrutura de Arquivos

In [ ]:
def listar_dataset(caminho_raiz: Path) -> pd.DataFrame:
    """Percorre o diretório e retorna um DataFrame com caminho, classe e split."""
    registros = []
    split_nome = caminho_raiz.name  # 'train', 'valid' ou 'test'

    for subdir in sorted(caminho_raiz.iterdir()):
        if not subdir.is_dir() or subdir.name not in MAPA_CLASSES:
            continue
        rotulo = MAPA_CLASSES[subdir.name]
        for arquivo in subdir.iterdir():
            if arquivo.suffix.lower() in ('.png', '.jpg', '.jpeg'):
                registros.append({
                    'caminho':    str(arquivo),
                    'classe_dir': subdir.name,
                    'rotulo':     rotulo,
                    'classe':     NOMES_CLASSES[rotulo],
                    'split':      split_nome,
                    'extensao':   arquivo.suffix.lower(),
                })

    return pd.DataFrame(registros)

df_treino = listar_dataset(CAMINHO_TREINO)
df_valid  = listar_dataset(CAMINHO_VALIDACAO)
df_teste  = listar_dataset(CAMINHO_TESTE)
df_total  = pd.concat([df_treino, df_valid, df_teste], ignore_index=True)

print(f'Total de imagens: {len(df_total)}')
print(f'  Treino    : {len(df_treino)}')
print(f'  Validação : {len(df_valid)}')
print(f'  Teste     : {len(df_teste)}')
df_total.head()

## 3. Distribuição de Classes

In [ ]:
# Contagem por split e classe
distribuicao = df_total.groupby(['split', 'classe']).size().unstack(fill_value=0)
distribuicao = distribuicao.loc[['train', 'valid', 'test']]  # Ordena splits
print('\nDistribuição de imagens por split e classe:')
display(distribuicao)

# Percentuais por split
print('\nPercentual por classe em cada split:')
display(distribuicao.div(distribuicao.sum(axis=1), axis=0).round(3) * 100)

In [ ]:
# Gráfico de barras agrupadas
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

splits      = [('train', df_treino, 'Treino'),
               ('valid', df_valid, 'Validação'),
               ('test',  df_teste, 'Teste')]
cores       = sns.color_palette('Set2', 4)

for ax, (split_key, df_split, titulo) in zip(axes, splits):
    contagem = df_split['classe'].value_counts()
    nomes_curtos = {
        'Adenocarcinoma':                'Adeno.',
        'Carcinoma de Células Grandes':  'Gr.Cell.',
        'Normal':                        'Normal',
        'Carcinoma de Células Escamosas':'Sq.Cell.',
    }
    contagem.index = [nomes_curtos.get(n, n) for n in contagem.index]

    barras = ax.bar(contagem.index, contagem.values, color=cores, edgecolor='white', linewidth=0.8)

    # Anota cada barra com a contagem
    for barra in barras:
        ax.text(
            barra.get_x() + barra.get_width() / 2,
            barra.get_height() + 1,
            str(int(barra.get_height())),
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )

    ax.set_title(f'{titulo} ({len(df_split)} imagens)', fontsize=12, pad=8)
    ax.set_xlabel('Classe', fontsize=10)
    ax.set_ylabel('Quantidade', fontsize=10)
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Distribuição de Classes por Split', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Amostras Visuais por Classe

In [ ]:
def mostrar_amostras(df: pd.DataFrame, n_por_classe: int = 4, titulo: str = '') -> None:
    """Exibe n imagens aleatórias para cada classe."""
    n_classes = df['classe'].nunique()
    fig, axes = plt.subplots(n_classes, n_por_classe, figsize=(n_por_classe * 3, n_classes * 3))

    for i, classe in enumerate(sorted(df['classe'].unique())):
        amostras = df[df['classe'] == classe].sample(n=n_por_classe, random_state=SEMENTE_ALEATORIA)

        for j, (_, linha) in enumerate(amostras.iterrows()):
            img = Image.open(linha['caminho']).convert('RGB')
            axes[i, j].imshow(img, cmap='gray')
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_ylabel(classe, fontsize=9, rotation=0,
                                      labelpad=70, va='center')
            axes[i, j].set_title(f'{img.size[0]}×{img.size[1]}', fontsize=8)

    plt.suptitle(titulo, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(f'../reports/amostras_{titulo.lower().replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()

mostrar_amostras(df_treino, n_por_classe=4, titulo='Amostras de Treino')

## 5. Análise das Dimensões das Imagens

In [ ]:
def coletar_dimensoes(df: pd.DataFrame, amostra: int = 200) -> pd.DataFrame:
    """Amostra imagens e retorna largura, altura e proporção."""
    df_amostrado = df.sample(n=min(amostra, len(df)), random_state=SEMENTE_ALEATORIA)
    dims = []
    for _, linha in df_amostrado.iterrows():
        try:
            with Image.open(linha['caminho']) as img:
                w, h = img.size
                dims.append({'largura': w, 'altura': h, 'proporcao': w / h,
                             'canais': len(img.getbands()), 'classe': linha['classe']})
        except Exception:
            pass
    return pd.DataFrame(dims)

df_dims = coletar_dimensoes(df_total, amostra=300)

print('Estatísticas de dimensão das imagens:')
display(df_dims[['largura', 'altura', 'proporcao']].describe().round(1))
print(f"\nCanais únicos: {df_dims['canais'].unique()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Scatter: largura × altura
axes[0].scatter(df_dims['largura'], df_dims['altura'], alpha=0.4, s=20)
axes[0].set_xlabel('Largura (px)')
axes[0].set_ylabel('Altura (px)')
axes[0].set_title('Dimensões das Imagens')
axes[0].axvline(224, color='red', linestyle='--', label='224px (alvo)')
axes[0].axhline(224, color='red', linestyle='--')
axes[0].legend()

# Histograma de proporções
axes[1].hist(df_dims['proporcao'], bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(1.0, color='red', linestyle='--', label='Quadrado (1:1)')
axes[1].set_xlabel('Proporção (Largura / Altura)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Proporções')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/dimensoes_imagens.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Análise de Intensidade de Pixels

In [ ]:
def histograma_intensidade(df: pd.DataFrame, n: int = 5) -> None:
    """Plota histograma de intensidade de pixel para n imagens de cada classe."""
    classes_unicas = sorted(df['classe'].unique())
    fig, axes = plt.subplots(1, len(classes_unicas), figsize=(14, 4))

    for ax, classe in zip(axes, classes_unicas):
        amostras = df[df['classe'] == classe].sample(n=min(n, len(df[df['classe'] == classe])),
                                                      random_state=SEMENTE_ALEATORIA)
        for _, linha in amostras.iterrows():
            img = np.array(Image.open(linha['caminho']).convert('L'))  # Escala de cinza
            ax.hist(img.ravel(), bins=50, alpha=0.4, density=True)

        ax.set_title(classe.split(' ')[0], fontsize=10)  # Nome curto
        ax.set_xlabel('Intensidade')
        if ax == axes[0]:
            ax.set_ylabel('Densidade')

    plt.suptitle('Histograma de Intensidade de Pixels por Classe', fontsize=12)
    plt.tight_layout()
    plt.savefig('../reports/histograma_intensidade.png', dpi=150, bbox_inches='tight')
    plt.show()

histograma_intensidade(df_treino, n=8)

## 7. Resumo e Conclusões

Com base na exploração acima, podemos concluir:

1. **Desbalanceamento de classes:** A classe `adenocarcinoma` tem mais amostras no treino (~195), enquanto `large.cell.carcinoma` tem apenas ~115. Estratégias como *weighted sampling* ou *class weights* na loss podem ser benéficas.

2. **Dimensões variadas:** As imagens possuem tamanhos diferentes — o redimensionamento para 224×224 (padrão ImageNet) é necessário.

3. **Imagens RGB:** Todas as imagens são convertíveis para 3 canais (RGB), compatíveis com os backbones pré-treinados.

4. **Volume de dados:** Com ~1.000 imagens no total, o uso de *Transfer Learning* e *Data Augmentation* é altamente recomendado para evitar overfitting.

**Próximo passo:** `02_preprocessamento.ipynb` — definir e validar o pipeline de pré-processamento.